In [2]:
import geopandas as gpd
import pandas as pd
import re
import os
from rapidfuzz import fuzz, process
from shapely import Point

#### Paths

In [3]:
names_dir = '../../../data/shapes'
farmers_xls = f'{names_dir}/farmers.xlsx'
shp_path = f'{names_dir}/awg_farmers_final.shp'
assert os.path.exists(farmers_xls), 'Check path'

#### Read in excel file and clean the headers

In [4]:
df = pd.read_excel(farmers_xls, skiprows=2, sheet_name='Sheet1')

In [5]:
df.head()

,Unnamed: 0,Unnamed: 1,Unnamed: 2,Unnamed: 3,Names of Famers,Trees given,Estimanted Number of\ntrees per acre,Spacing,Species,Acreage,Coordinates
0,NaN,NaN,NaN,1,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E"
1,NaN,NaN,NaN,2,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E"
2,NaN,NaN,NaN,3,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E"
3,NaN,NaN,NaN,4,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E"
4,NaN,NaN,NaN,5,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E"


In [6]:
df = df.iloc[:,4:]

In [7]:
df.columns = df.columns.str.strip().str.replace(' ', '_').str.lower()

In [8]:
df.columns

Index(['names_of_famers', 'trees_given',
       'estimanted_number_of\ntrees_per_acre', 'spacing', 'species', 'acreage',
       'coordinates'],
      dtype='str')

In [9]:
df.rename(columns={'estimanted_number_of\ntrees_per_acre': 'trees_per_acre', 'names_of_famers' : 'name'}, inplace=True )

#### Convert the coordinates from deg min secs to decimal degrees

In [9]:
def parse_dms(dms_str):
    if pd.isna(dms_str): return None, None
    
    # Matches: degrees° minutes' seconds" direction
    pattern = r"(\d+(?:\.\d+)?)°(\d+(?:\.\d+)?)\'(\d+(?:\.\d+)?)\"([NSEW])"
    parts = re.findall(pattern, str(dms_str))
    
    def to_dd(deg, mn, sec, direction):
        dd = float(deg) + float(mn)/60 + float(sec)/3600
        if direction in ['S', 'W']:
            dd *= -1
        return dd

    # Returns (Latitude, Longitude) as floats
    if len(parts) == 2:
        lat = to_dd(*parts[0])
        lon = to_dd(*parts[1])
        return lat, lon
    return None, None

# Usage with your Excel file
df[['lat', 'lon']] = df['coordinates'].apply(lambda x: pd.Series(parse_dms(x)))

In [10]:
df.head()

,name,trees_given,trees_per_acre,spacing,species,acreage,coordinates,lat,lon
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028


In [11]:
df.dropna(subset=['lat', 'lon'], inplace=True)

In [12]:
df['estimated_trees_present'] = round(df['trees_per_acre'] * df['acreage'], ndigits=1)

In [13]:
df['estimated_trees_present'].describe()

count     163.000000
mean      192.797546
std       156.934175
min       100.000000
25%       150.000000
50%       150.000000
75%       150.000000
max      1000.000000
Name: estimated_trees_present, dtype: float64

#### Create geometry column using shapely

In [14]:
geometry = [Point(xy) for xy in zip(df['lon'], df['lat'])]

In [15]:
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

In [16]:
gdf.head()

,name,trees_given,trees_per_acre,spacing,species,acreage,coordinates,lat,lon,estimated_trees_present,geometry
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167,380.0,POINT (37.86517 -1.58233)
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694,150.0,POINT (37.88069 -1.5395)
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639,230.0,POINT (37.86064 -1.61897)
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306,150.0,POINT (37.87131 -1.62325)
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028,630.0,POINT (37.86903 -1.61944)


In [17]:
gdf.isna().sum()

name                       0
trees_given                0
trees_per_acre             0
spacing                    0
species                    0
acreage                    0
coordinates                0
lat                        0
lon                        0
estimated_trees_present    0
geometry                   0
dtype: int64

In [18]:
len(gdf)

163

In [19]:
# gdf.to_file(f'{names_dir}/clean_file.fgb')

### Merge with the shapefile

In [20]:
final_gdf = gpd.read_file(shp_path)

In [21]:
final_gdf.head()

,kamiti_cbo,awg_kml,trees_give,trees_aliv,melia_only,bounds_ok,geometry
0,Naomi Nicodemus Mutuku,Naomie Nicodemus Mutuku,121.0,152.0,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,..."
1,Emma Mutie,Emma Mutie,150.0,145.0,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249..."
2,Dorrine Kasyoka Mutua,Dorrine kasyoka Mutua,108.0,115.0,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,..."
3,Kinyao Musembi,Kinyao Musembi,150.0,33.0,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779..."
4,Simon Mumo Mulu,Simon mumo Mulu,150.0,37.0,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017..."


In [22]:
final_gdf.drop(columns=['kamiti_cbo'], inplace=True)

In [23]:
final_gdf.rename(columns={'awg_kml' : 'name'}, inplace=True)

In [24]:
len(final_gdf)

166

#### Name normalization

In [25]:
# Define unwanted substrings / tokens
UNWANTED_TERMS = [
    "awg",
    "farmer",
    "old farmer",
    "new",
    "ond",
    '-'
]

In [26]:
def normalize_name(name):
    if pd.isnull(name):
        return None
    name = name.lower()
    name = re.sub(r'[^\w\s]', '', name)  # remove punctuation
    for term in sorted(UNWANTED_TERMS, key=len, reverse=True):
        pattern = r'\b' + re.escape(term) + r'\b'
        name = re.sub(pattern, '', name)
    
    # Remove standalone 4-digit years
    name = re.sub(r'\b\d{4}\b', '', name)
    name = re.sub(r'\s+', ' ', name)     # collapse spaces
    return name.strip()

In [27]:
# Remove null names
df = df[df["name"].notnull()]

# Normalize
df["name_clean"] = df["name"].apply(normalize_name)

# Remove empty
df = df[df["name_clean"].notnull()]

In [28]:
# Remove null names
final_gdf = final_gdf[final_gdf["name"].notnull()]

# Split grouped names
final_gdf["name_split"] = final_gdf["name"].str.split(",")

# Explode into individual rows
final_gdf = final_gdf.explode("name_split")

# Normalize
final_gdf["name_clean"] = final_gdf["name_split"].apply(normalize_name)

# Remove empty
final_gdf = final_gdf[final_gdf["name_clean"].notnull()]

In [29]:
len(df)

163

In [30]:
len(final_gdf)

167

In [31]:
final_gdf.loc[final_gdf['geometry'].duplicated()]

,name,trees_give,trees_aliv,melia_only,bounds_ok,geometry,name_split,name_clean
14,Maurice mulaa Manzolo,250.0,250.0,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",Maurice mulaa Manzolo,maurice mulaa manzolo
15,Rose martha Mulaa,350.0,350.0,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",Rose martha Mulaa,rose martha mulaa
16,joel nyamai mulaa,NaN,NaN,T,T,"POLYGON ((37.86484 -1.58302, 37.86482 -1.58301...",joel nyamai mulaa,joel nyamai mulaa
23,Wayua Singila,200.0,105.0,T,T,"POLYGON ((37.85664 -1.58264, 37.85687 -1.58272...",Wayua Singila,wayua singila
55,robert Kasoa Munyilu,150.0,22.0,T,T,"POLYGON ((37.87673 -1.55065, 37.87681 -1.55057...",robert Kasoa Munyilu,robert kasoa munyilu
150,Priska martha Mutia,150.0,156.0,T,T,"POLYGON ((37.87066 -1.62294, 37.87081 -1.62283...",Priska martha Mutia,priska martha mutia
162,"jeremiah ngambi mutisya, ruth mumbua willie",NaN,NaN,T,F,"POLYGON ((37.86815 -1.61886, 37.8681 -1.61794,...",ruth mumbua willie,ruth mumbua willie


In [111]:
df.head()

,name,trees_given,trees_per_acre,spacing,species,acreage,coordinates,lat,lon,estimated_trees_present,name_clean
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167,380.0,rose martha mulaa
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694,150.0,john kithuku mulu
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639,230.0,stella kavindu isaac
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306,150.0,priscah martha mutia
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028,630.0,daina nditi mutisya ngambi


#### Matching names

In [62]:
def get_fuzzy_matches(df1, df2, col1, col2, threshold=70):
    """
    Finds the best match from df2 for every name in df1.
    """
    choices = df2[col2].dropna().tolist()
    
    # We use extractOne to find the single best match for each name
    matches = []
    for name in df1[col1]:
        # 1. Handle potential NaN in the name itself
        if pd.isna(name):
            matches.append(None)
            continue

        # 2. Get the result without immediate unpacking
        result = process.extractOne(name, choices, scorer=fuzz.token_sort_ratio, score_cutoff=threshold)
        
        # 3. Check if a match was actually found
        if result:
            best_match, score = result[0], result[1] # Safe manual unpack
            if score >= threshold:
                matches.append(best_match)
            else:
                matches.append(None)
        else:
            matches.append(None)
    return matches

In [63]:
# --- STEP 2: Create the Inspection Dataframe ---
# Let's assume your name columns are both called 'farmer_name'
final_gdf['matched_name'] = get_fuzzy_matches(final_gdf, df, 'name_clean', 'name_clean')

In [74]:
nzioka = final_gdf.iloc[138]

In [75]:
nzioka

name                                                Esther Nzioka
trees_give                                                  200.0
trees_aliv                                                   76.0
melia_only                                                      T
bounds_ok                                                       T
geometry        POLYGON ((37.879801787152026 -1.59845304603782...
name_split                                          Esther Nzioka
name_clean                                          esther nzioka
matched_name                                        esther nzioka
Name: 138, dtype: object

In [ ]:
# final_gdf.to_csv(f'{names_dir}/check.csv')

#### Merge the 2 dfs

In [81]:
df.head()

,name,trees_given,trees_per_acre,spacing,species,acreage,coordinates,lat,lon,estimated_trees_present,name_clean
0,Rose Martha Mulaa,380,450,3 m x 3 m,Melia volkensii,0.844444,"1°34'56.4""S 37°51'54.6""E",-1.582333,37.865167,380.0,rose martha mulaa
1,John Kithuku Mulu,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°32'22.2""S 37°52'50.5""E",-1.539500,37.880694,150.0,john kithuku mulu
2,Stella Kavindu Isaac,230,450,3 m x 3 m,Melia volkensii,0.511111,"1°37'08.3""S 37°51'38.3""E",-1.618972,37.860639,230.0,stella kavindu isaac
3,Priscah Martha Mutia,150,450,3 m x 3 m,Melia volkensii,0.333333,"1°37'23.7""S 37°52'16.7""E",-1.623250,37.871306,150.0,priscah martha mutia
4,Daina Nditi Mutisya Ngambi,630,450,3 m x 3 m,Melia volkensii,1.400000,"1°37'10.0""S 37°52'08.5""E",-1.619444,37.869028,630.0,daina nditi mutisya ngambi


In [82]:
df.columns

Index(['name', 'trees_given', 'trees_per_acre', 'spacing', 'species',
       'acreage', 'coordinates', 'lat', 'lon', 'estimated_trees_present',
       'name_clean'],
      dtype='str')

In [83]:
final_gdf.columns

Index(['name', 'trees_give', 'trees_aliv', 'melia_only', 'bounds_ok',
       'geometry', 'name_split', 'name_clean', 'matched_name'],
      dtype='str')

In [87]:
final_gdf = final_gdf.merge(df[['trees_given', 'species',
       'estimated_trees_present', 'name_clean']], on='name_clean', how='left')

In [86]:
final_gdf['trees_give'].info()

<class 'pandas.Series'>
Index: 167 entries, 0 to 165
Series name: trees_give
Non-Null Count  Dtype  
--------------  -----  
137 non-null    float64
dtypes: float64(1)
memory usage: 2.6 KB


In [88]:
final_gdf['trees_give'] = final_gdf['trees_give'].fillna(final_gdf['trees_given'])

In [89]:
final_gdf['trees_give'].info()

<class 'pandas.Series'>
RangeIndex: 167 entries, 0 to 166
Series name: trees_give
Non-Null Count  Dtype  
--------------  -----  
149 non-null    float64
dtypes: float64(1)
memory usage: 1.4 KB


In [90]:
final_gdf.columns

Index(['name', 'trees_give', 'trees_aliv', 'melia_only', 'bounds_ok',
       'geometry', 'name_split', 'name_clean', 'matched_name', 'trees_given',
       'species', 'estimated_trees_present'],
      dtype='str')

In [92]:
final_gdf = final_gdf.drop(columns=['name', 'trees_given', 'name_split', 'matched_name', 'estimated_trees_present'])

In [94]:
final_gdf.rename(columns={'trees_give' : 'trees_given', 'trees_aliv' : 'trees_alive', 'name_clean' : 'name'}, inplace=True)

In [95]:
final_gdf.head()

,trees_given,trees_alive,melia_only,bounds_ok,geometry,name,species
0,121.0,152.0,T,T,"POLYGON ((37.85681 -1.5947, 37.85699 -1.59471,...",naomie nicodemus mutuku,NaN
1,150.0,145.0,T,T,"POLYGON ((37.85454 -1.59251, 37.85467 -1.59249...",emma mutie,Melia volkensii
2,108.0,115.0,F,T,"POLYGON ((37.85088 -1.59412, 37.8509 -1.59402,...",dorrine kasyoka mutua,Melia volkensii
3,150.0,33.0,F,T,"POLYGON ((37.86475 -1.58786, 37.86461 -1.58779...",kinyao musembi,Melia volkensii
4,150.0,37.0,F,T,"POLYGON ((37.87997 -1.59015, 37.88011 -1.59017...",simon mumo mulu,Melia volkensii


In [97]:
final_gdf = final_gdf[['name', 'trees_given', 'trees_alive', 'melia_only', 'bounds_ok', 'species', 'geometry']]

In [112]:
final_gdf = final_gdf.to_crs('EPSG:32737')

In [105]:
final_gdf['name'] = final_gdf['name'].str.title()

In [113]:
final_gdf.head()

,name,trees_given,trees_alive,melia_only,bounds_ok,species,geometry
0,Naomie Nicodemus Mutuku,121.0,152.0,True,True,NaN,"POLYGON ((372831.949 9823702.107, 372851.714 9..."
1,Emma Mutie,150.0,145.0,True,True,Melia volkensii,"POLYGON ((372578.994 9823944.388, 372594.168 9..."
2,Dorrine Kasyoka Mutua,108.0,115.0,False,True,Melia volkensii,"POLYGON ((372171.895 9823766.202, 372174.053 9..."
3,Kinyao Musembi,150.0,33.0,False,True,Melia volkensii,"POLYGON ((373715.511 9824459.051, 373699.469 9..."
4,Simon Mumo Mulu,150.0,37.0,False,True,Melia volkensii,"POLYGON ((375408.007 9824206.071, 375424.481 9..."


In [103]:
final_gdf['melia_only'] = final_gdf['melia_only'].map({'T': True, 'F': False})
final_gdf['bounds_ok'] = final_gdf['bounds_ok'].map({'T': True, 'F': False})

In [107]:
final_gdf.info()

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 167 entries, 0 to 166
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   name         167 non-null    str     
 1   trees_given  149 non-null    float64 
 2   trees_alive  135 non-null    float64 
 3   melia_only   167 non-null    bool    
 4   bounds_ok    167 non-null    bool    
 5   species      97 non-null     str     
 6   geometry     167 non-null    geometry
dtypes: bool(2), float64(2), geometry(1), str(2)
memory usage: 7.0 KB


In [114]:
final_gdf.to_file(f'{names_dir}/awg_farmers_boundaries.shp')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_13752\1399428328.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  final_gdf.to_file(f'{names_dir}/awg_farmers_boundaries.shp')
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'trees_given' to 'trees_give'
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'trees_alive' to 'trees_aliv'
  ogr_write(


In [115]:
final_gdf.to_csv(f'{names_dir}/awg_farmers_boundaries.csv')

#### After adding tree audit data to records that had none, import the corrected csv again

In [22]:
# Read as a regular DF first
corrected = pd.read_csv(f'{names_dir}/awg_farmers_boundaries.csv')

# Convert the text column to actual geometry objects
corrected['geometry'] = gpd.GeoSeries.from_wkt(corrected['geometry']) 

# Cast to GeoDataFrame
corrected_gdf = gpd.GeoDataFrame(corrected, geometry='geometry', crs="EPSG:32737")

In [19]:
corrected.head()

,field_1,name,trees_given,trees_alive,melia_only,bounds_ok,species,geometry
0,0,Naomie Nicodemus Mutuku,121,152,TRUE,TRUE,,POLYGON ((372831.94878690824 9823702.107416678...
1,1,Emma Mutie,150,145,TRUE,TRUE,Melia volkensii,"POLYGON ((372578.9943232875 9823944.388224218,..."
2,2,Dorrine Kasyoka Mutua,108,115,FALSE,TRUE,Melia volkensii,"POLYGON ((372171.8945287593 9823766.201867847,..."
3,3,Kinyao Musembi,150,33,FALSE,TRUE,Melia volkensii,"POLYGON ((373715.5113339135 9824459.051429205,..."
4,4,Simon Mumo Mulu,150,37,FALSE,TRUE,Melia volkensii,POLYGON ((375408.00692821614 9824206.071153713...


In [23]:
corrected_gdf.to_file(f'{names_dir}/awg_farmers_boundaries.shp')

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_4156\673908118.py:1: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  corrected_gdf.to_file(f'{names_dir}/awg_farmers_boundaries.shp')
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'Unnamed: 0' to 'Unnamed_ 0'
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'trees_given' to 'trees_give'
  ogr_write(
e:\acre_insights\regreening_project\venv\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'trees_alive' to 'trees_aliv'
  ogr_write(


In [24]:
corrected_gdf.to_excel(f'{names_dir}/awg_farmers_boundaries.xlsx')